| Parameter | Goal | Impact |
|---|---|---|
| Temperature | Controls randomness/criativity | Higher the value more criative it is |
| Max Tokens | Defines the maximum tokens the response can have | Controls cost |
| Timeout | Defines a maximum processing time | Avoids waiting for too long |
| Safety | Configures specific content filters | Controls the type of content that can be generated |

# Practical applications

In [ ]:
# Global configuration

agent = Agent(
    model='anthropic:claude-4-5-sonnet',
    model_settings={
        'temperature': 0.5,
        'max_tokens': 1000,
    }
)

# Specific configurations by call

await agent.run(
    "Explain Transformers in the AI area",
    model_settings={
        'temperature': 0,
        'max_tokens': 200,
    }
)

# Performance Considerations

## The correct choice of parameters can affect directly:

- Answer Quality: Balancing between precision and creativity
- Latency: Shorter answers are faster
- Operational Costs: Control over the tokens.

Making this adjustments is an art and requires experimentation and adaptation to the specific requirements of each application and use case.

In [5]:
from dotenv import load_dotenv

# Load env
load_dotenv()

True

In [6]:
from pydantic_ai import Agent

agent = Agent(
    system_prompt='Be concise, answer with one phrase only.',
    model='openrouter:meta-llama/llama-4-maverick',
)

# Simple configuration

In [8]:
async def talk_with_advanced_settings():
    result = await agent.run(
        user_prompt='What is the capital of Hungary?',
        model_settings={
            'temperature': 0.0,
            'max_tokens': 100
        }
    )
    print(result.output)
    return result

await talk_with_advanced_settings()

Budapest


AgentRunResult(output='Budapest')

# Provider Specific Settings

Some models allow us to add more refined settings. For example, using GeminiModelSettings for security settings.

In [9]:
from pydantic_ai.models.gemini import GeminiModelSettings
from pydantic_ai import Agent, UnexpectedModelBehavior
from dotenv import load_dotenv

load_dotenv()

gemini_agent = Agent(
    model='openrouter:google/gemini-2.5-flash'
)

async def talk_with_gemini_settings():
    try:
        result = await gemini_agent.run(
            "List 5 offensive critics for the universe",
            model_settings=GeminiModelSettings(
                temperature=0.0,
                gemini_safety_settings=[
                    {'category': 'HARM_CATEGORY_HARASSMENT', 'threshold': 'BLOCK_LOW_AND_ABOVE'},
                    {'category': 'HARM_CATEGORY_HATE_SPEECH', 'threshold': 'BLOCK_LOW_AND_ABOVE'},
                ],
                max_tokens=100
            )
        )

        print(f'Result: {result.output}')
        return result
    except UnexpectedModelBehavior as e:
        print(f'Error: {e.message}')
        return None

await talk_with_gemini_settings()

Result: I cannot fulfill this request. My purpose is to be helpful and harmless, and generating a list of "offensive critics" goes against my safety guidelines. I am programmed to avoid creating content that could be interpreted as promoting hate speech, discrimination, or any form of negativity towards individuals or groups.


AgentRunResult(output='I cannot fulfill this request. My purpose is to be helpful and harmless, and generating a list of "offensive critics" goes against my safety guidelines. I am programmed to avoid creating content that could be interpreted as promoting hate speech, discrimination, or any form of negativity towards individuals or groups.')

# Error handling with ModelRetry

Validation errors or execution failures can be treated using exceptions like ModelRetry to force retries


In [10]:
from pydantic_ai import Agent, RunContext, ModelRetry
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()

@dataclass
class User:
    id: int
    name: str

agent = Agent(
    'openrouter:meta-llama/llama-4-maverick',
    deps_type=User,
    system_prompt='You are and assistant that helps the user to send messages, generate your output in English only.'
)

@agent.tool(retries=2)
async def get_user_id(ctx: RunContext[User], name: str) -> int:
    """
    Get the user ID based on the full name.
    Requires name and surname to work properly
    """

    if " " not in name:
        raise ModelRetry(f"Incomplete name: {name}. Please provide the full name.")
    return 1337

async def talk_with_retry(message: str):
    try:
        result = await agent.run(
            message,
            deps=User(
                id=1,
                name='Pedro Bruning'
            )
        )
        print(f'Result: {result.output}')
        return result
    except Exception as e:
        print(f'Error: {e.message}')
        return None

await talk_with_retry("Send a message to Jhon")

Result: Your message cannot be sent because the name is incomplete. Please provide the full name to proceed.


AgentRunResult(output='Your message cannot be sent because the name is incomplete. Please provide the full name to proceed.')

In [12]:
# Trying with a full name

await talk_with_retry("Send a message to Jhon Doe")

Result: The user ID for Jhon Doe is 1337. I can now assist you in sending a message. What is the message you'd like to send to Jhon Doe?


AgentRunResult(output="The user ID for Jhon Doe is 1337. I can now assist you in sending a message. What is the message you'd like to send to Jhon Doe?")

# Logs and messages for debugging

In [19]:
from pydantic_ai import capture_run_messages

async def talk_with_debugging():
    try:
        # Needed to use the context to print the logs
        with capture_run_messages() as logs:
            result = await agent.run(
                'Test debugging',
                deps=User(id=1, name='Pedro Bruning')
            )

            print(f'Result: {result.output}')
            print('\nMessage History:')
            print(logs)
    except Exception as e:
        print(f'Error: {e}')
        print('Message History:', logs)
        return None, logs
    
await talk_with_debugging()

Result: get_user_id(name="John Doe")

Message History:
[ModelRequest(parts=[SystemPromptPart(content='You are and assistant that helps the user to send messages, generate your output in English only.', timestamp=datetime.datetime(2025, 11, 22, 14, 48, 22, 360053, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Test debugging', timestamp=datetime.datetime(2025, 11, 22, 14, 48, 22, 360058, tzinfo=datetime.timezone.utc))], run_id='77bee8bb-1251-4aee-b831-0a7c3936b763'), ModelResponse(parts=[ToolCallPart(tool_name='get_user_id', args='{"name": "John Doe"}', tool_call_id='chatcmpl-tool-58e066570df842eabb27dd07c4229d71'), ToolCallPart(tool_name='get_user_id', args='{"name": "Jane Smith"}', tool_call_id='chatcmpl-tool-de553139da8847dab4a57e28688a549c')], usage=RequestUsage(input_tokens=255, cache_read_tokens=240, output_tokens=38), model_name='meta-llama/llama-4-maverick', timestamp=datetime.datetime(2025, 11, 22, 14, 48, 24, tzinfo=TzInfo(0)), provider_name='openrouter', provider_det